<a href="https://colab.research.google.com/github/frasercrichton/ai-dde-hackthon/blob/feature%2Fleiden-guidelines-doc/leiden-guidelines-doc/team-red/notebooks/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# from transformers import AutoTokenizer, AutoModel
from langchain.text_splitter import RecursiveCharacterTextSplitter
# import torch

from google.colab import userdata
userdata.get('GITHUB_TOKEN')

!git clone https://{GITHUB_TOKEN}@github.com/frasercrichton/ai-dde-hackthon.git
%cd ai-dde-hackthon
! git checkout feature/leiden-guidelines-doc
%cd team-red
! ls
! pip install chromadb

Cloning into 'ai-dde-hackthon'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 403 (delta 130), reused 88 (delta 51), pack-reused 197 (from 1)
Receiving objects: 100% (403/403), 4.15 MiB | 8.69 MiB/s, done.
Resolving deltas: 100% (209/209), done.
/content/ai-dde-hackthon/team-red/ai-dde-hackthon
Branch 'feature/leiden-guidelines-doc' set up to track remote branch 'feature/leiden-guidelines-doc' from 'origin'.
Switched to a new branch 'feature/leiden-guidelines-doc'
/content/ai-dde-hackthon/team-red/ai-dde-hackthon/team-red
data		   images	   README.md	     test
data-cleaning	   notebooks	   requirements.txt  training
graph_config.yaml  pyproject.toml  src


In [12]:
import sys
from pathlib import Path

cwd = Path.cwd()
project_root = f'{cwd.parent}'

sys.path.append(project_root)

print('The project directory is:', project_root)
print('The working directory is:', cwd)

The project directory is: /content/ai-dde-hackthon/team-red/ai-dde-hackthon
The working directory is: /content/ai-dde-hackthon/team-red/ai-dde-hackthon/team-red


In [13]:
import logging

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(name)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
#
# self.logger = logging.getLogger(self.__class__.__name__)

logger = logging.getLogger(__name__)

In [14]:
from curses import meta
import chromadb

class RAGDatabase:

    client = None

    def __init__(self, collection_name):
        logger.info('Initialising.')

        if RAGDatabase.client is None:
            logger.info('RAGDatabase.client is none.')
            RAGDatabase.client = chromadb.Client()
            self.client = RAGDatabase.client

        # logger.info('Resetting.')
        # self.client.reset()
        self.collection = self.client.get_or_create_collection(name=collection_name)

    def store_documents(self, documents: list):
        kwargs = {
            'documents': [doc.get('text') for doc in documents],
            'embeddings': [doc.get('embedding') for doc in documents],
            'ids': [doc.get('id') for doc in documents]
        }
        # [{}, {}, {}, {}]
        metadata = [doc.get('metadata') for doc in documents if doc.get('metadata')]
        if len(metadata) > 0:
            kwargs['metadatas'] = metadata

        try:
            self.collection.upsert(**kwargs)
        except Exception as e:
            logging.error(f'Error storing documents: {e}')

    def get_collection(self, collection_name):
        collection = self.client.get_collection(collection_name)
        return collection.get()

    def query_with_embeddings(self, embedding_query, metadata_query=None, n_results=3):

        query = {
            'query_embeddings': embedding_query,
            'n_results': n_results
        }

        if metadata_query:
            query['where'] = metadata_query


        results = self.collection.query(**query)

        # logger.info(results)

        # Here’s the logic for this:
        #   •	Lower distances indicate higher similarity (the documents are more relevant).
        #   •	Higher distances indicate lower similarity (the documents are less relevant).
        threshold = 50.0
        distances = results['distances'][0]

        if not distances or min(distances) > threshold:
            logger.info(f'No relevant documents found (Distances {distances}).')
            return [{
                'text': 'No relevant documents found for this query',
                'id': 'unknown',
                'metadata': {}
            }]
        else:
            logger.info(f'Proceed with RAG... {distances}')

        return [
            {
                'text': doc_text,
                'id': doc_id,
                'metadata': doc_metadata
            }
            for doc_text, doc_id, doc_metadata in zip(results['documents'][0], results['ids'][0], results['metadatas'][0])
        ]


    def query_with_text(self, query: str, filter, n_results=3):
            # TODO you can query by a list
        results = self.collection.query(
            query_texts=[query], where=filter, n_results=n_results
        )
        return [
            {
                'text': doc_text,
                'id': results['ids'][0][i],
                'metadata': results['metadatas'][0][i],
            }
            for i, doc_text in enumerate(results['documents'][0])
        ]

    def delete_collection(self, collection_name):
        self.client.delete_collection(collection_name)

    def reset_database(self):
        self.client.reset()

In [15]:
from src.metadata_query import MetadataQuery

metadata_query = MetadataQuery().createFilters(headers=['VIDEOS'])
print(metadata_query)

metadata_query = MetadataQuery().createFilters(headers=['VIDEOS', 'PHOTOGRAPHS'])
print(metadata_query)

metadata_query = MetadataQuery().createFilters(headers=['VIDEOS'], tags=['working language', 'court language'])
print(metadata_query)


{'headers': 'VIDEOS'}
{'$or': [{'headers': {'$in': ['VIDEOS', 'PHOTOGRAPHS']}}]}
{'$or': [{'headers': {'$in': ['VIDEOS']}}, {'working language': True}, {'court language': True}]}


In [16]:
from src.document_parser import DocumentParser
from src.embeddings_processor import EmbeddingsProcessor
from src.tokenizer import Tokenizer

rag_database = RAGDatabase(collection_name='test')

tokeniser = Tokenizer('sentence-transformers/all-MiniLM-L6-v2')

embeddings_processor = EmbeddingsProcessor('sentence-transformers/all-MiniLM-L6-v2', tokeniser)
document_parser = DocumentParser()

documents =[
    {'text': 'The Eiffel Tower is located in Paris, France.'},
    {'text': 'The Great Wall of China is one of the Seven Wonders of the World.'},
    {'text': 'Python is a popular programming language for data science.'},
    {'text': 'Leonardo da Vinci painted the Mona Lisa.'},
]

preprocessed_documents = document_parser.preprocess_documents_for_chroma(documents)
logger.info(preprocessed_documents)


# documents_with_embeddings = document_parser.create_embedded_documents(tokenised_documents, embeddings_processor)
# print(documents_with_embeddings)

# query_embeddings = embeddings_processor.create_embeddings(tokenised_documents)
# print(query_embeddings)

documents_with_embeddings = document_parser.create_embedded_documents(preprocessed_documents, embeddings_processor)
rag_database.store_documents(documents_with_embeddings)


2025-05-11 14:21:32 - INFO - __main__ - Initialising.
2025-05-11 14:21:32 - INFO - __main__ - RAGDatabase.client is none.
2025-05-11 14:21:32 - INFO - __main__ - [{'metadata': {}, 'text': 'The Eiffel Tower is located in Paris, France.'}, {'metadata': {}, 'text': 'The Great Wall of China is one of the Seven Wonders of the World.'}, {'metadata': {}, 'text': 'Python is a popular programming language for data science.'}, {'metadata': {}, 'text': 'Leonardo da Vinci painted the Mona Lisa.'}]
2025-05-11 14:21:32 - INFO - src.document_parser - parsing 4 documents.
2025-05-11 14:21:32 - INFO - src.embeddings_processor - cuda available
2025-05-11 14:21:32 - INFO - src.embeddings_processor - cuda available
2025-05-11 14:21:32 - INFO - src.embeddings_processor - cuda available
2025-05-11 14:21:32 - INFO - src.embeddings_processor - cuda available


In [17]:
# question = Tokenizer('sentence-transformers/all-MiniLM-L6-v2').tokenize()
query_embeddings = embeddings_processor.create_embeddings('who painted the Mona Lisa?')
x = rag_database.query_with_embeddings(query_embeddings)
logger.info(f'search result {x}')


2025-05-11 14:21:33 - INFO - src.embeddings_processor - cuda available
2025-05-11 14:21:33 - INFO - __main__ - Proceed with RAG... [14.702789306640625, 64.80664825439453, 74.11920928955078]
2025-05-11 14:21:33 - INFO - __main__ - search result [{'text': 'Leonardo da Vinci painted the Mona Lisa.', 'id': 'None-3', 'metadata': None}, {'text': 'The Great Wall of China is one of the Seven Wonders of the World.', 'id': 'None-1', 'metadata': None}, {'text': 'The Eiffel Tower is located in Paris, France.', 'id': 'None-0', 'metadata': None}]


In [18]:
rag_database = RAGDatabase(collection_name='test')

documents = [
    {
        'text': 'The Eiffel Tower is located in Paris, France.',
        'metadata': {
            'headers': 'architecture',
            'tags': [
                'france',
                'architecture',
                'Paris',
            ],
        },
    },
    {
        'text': 'The Great Wall of China is one of the Seven Wonders of the World.',
        'metadata': {
            'headers': 'architecture',
            'tags': [
                'china',
                'architecture',
            ],
        },
    },
    {
        'text': 'Python is a popular programming language for data science.',
        'metadata': {
            'headers': 'programming',
            'tags': [
                'programming',
            ],
        },
    },
    {
        'text': 'Leonardo da Vinci painted the Mona Lisa.',
        'metadata': {
            'headers': 'architecture',
            'tags': [
                'france',
                'painting'
            ],
        },
    },
]


embeddings_processor = EmbeddingsProcessor('sentence-transformers/all-MiniLM-L6-v2')
chroma_documents_x = DocumentParser().format_documents_for_chroma(documents_x)
print(f'********** {chroma_documents_x}')
rag_database.store_documents(chroma_documents_x)


question = Tokenizer('sentence-transformers/all-MiniLM-L6-v2').tokenize('who painted the Mona Lisa?')


query_embeddings = embeddings_processor.create_embeddings(question)


metadata_query = MetadataQuery().createFilters(headers=['x'], tags=['france'])

x = rag_database.query_with_embeddings(query_embeddings, metadata_query)
logger.info(f'search result {x}')

2025-05-11 14:21:33 - INFO - __main__ - Initialising.


TypeError: EmbeddingsProcessor.__init__() missing 1 required positional argument: 'tokenizer'

In [ ]:

leiden_documents = [
  {
  'metadata': {
   'headers': 'VIDEOS',
    'tags': [
      'no excerpts',
      'entire video',
      'complete footage',
      'full recording'
    ]},
    'text': 'Submission of videos in full, alongside their respective transcripts and translations, assist the Court in contextualising the segments of the video that have been identified as most relevant by the tendering party.30 The ICC Trial Chamber in Ntaganda admitted a full video broadcast instead of only the excerpts submitted by the Defence in order to provide context to the security situation portrayed by the video in its entirety.31 Excerpts. If, nevertheless, a party seeks to tender excerpts, the tendering party should also clearly indicate whether the full footage was available and who extracted the segments of the video.32 The opposing party may tender additional excerpts to assist the Court in contextualising the segments sought to be admitted.33 The ICC Trial Chamber in Ntaganda granted the Prosecution’s request to admit extensions of video excerpts that had been tendered by the Defence in order to illustrate the reason behind the presence of community leaders at an event depicted in the video excerpts'
  },
  {
  'metadata': {

   "headers": "VIDEOS",
    "tags": [
      'translation',
      'integral evidence',
      'associated materials',
      'combined evidence'
    ]},
    'text': 'Transcript and translation documents are written records designed to faithfully reflect the contents of the video for better comprehension.35 Consequently, each document and the video are treated as parts of the same evidence. The formal submission of a video automatically includes recognising the formal submission of associated transcripts and translations which have been duly disclosed.36 Similarly, it would be inconsistent to impose restrictions on one part but not the others.37 The ICC Trial Chamber in Katanga and Ngudjolo Chui granted a request to apply the same protective measures to the transcript and translation of a video that the Prosecution had been authorised to apply to the video itself.38 To facilitate the presentation of the evidence in court, the tendering party should, as early as practicable, indicate the segments of the video, transcript, and translation which it intends to use.39 The parties should also consult and resolve any disagreements about the transcripts or translations.40 No transcript is necessary if the purpose of the video is to demonstrate ambient sound;41 the Defence in Mladić at the ICTY was not required to transcribe the ambient sound of a firefight in a video it tendered.'
  },
  {
    'metadata': {

    "headers": "VIDEOS",
    "tags": [
      "working language",
      "court language",
      "submission deadline",
      "chamber access"
    ]},
    "text": "Translation. Pursuant to Regulation 39(1) of the Regulations of the Court, all documents and materials filed with the Registry shall be in a working language of the Court. If segments of the video are not in a working language of the Court, those segments must be translated into a working language of the Court before they can be deemed admissible.43 The Prosecution has not complied with its disclosure obligations under Rule 77 of the ICC Rules of Procedure and Evidence until the translations have been provided to the Defence.44 The translation requirement is based on the accused's right to be informed of the evidence upon which the Prosecution intends to rely, including the nature, cause and content of the charge.45 Moreover, the Chamber must be in a position to fully understand the evidence upon which the parties intend to rely.46 Accuracy of Translation. Videos must be of a sufficient sound quality to facilitate translation. In Mladić, the ICTY Trial Chamber relied upon a video’s English/French subtitles ‘in order not to get stuck’, although the sound quality in the Bosnian/Croatian/Serbian (BCS) original of the video was so poor that it risked inaccurate translation.47 The Defence, however, was permitted to rely upon this video so the proceedings could continue, but it was instructed to find a better BCS version of the video.48 In the absence of a coherent and intelligible version"
  },
  {
      'metadata': {

    "headers": "PHOTOGRAPHS",
    "tags": [
      "content inference",
      "visual analysis",
      "proof from image"
    ]},
    "text": 'This Guideline is derived from the ICC’s treatment of video evidence, but it can reasonably be applied to photographs as well. Caution should be exercised when considering a photograph since differences in personal perception can cause difficulties in making a definite finding.93 The Court will rely on the photograph only to the extent that it can make such a definite finding.94 The ICC Trial Chamber in Lubanga found that a reliable distinction can be drawn between individuals of different ages, based solely on the individuals’ appearance.95 Pursuant to Rule 63(4) of the ICC Rules of Procedure and Evidence, there is no strict legal requirement that a photograph has to be corroborated by other evidence for the Court to be able to rely on it and establish a specific fact'
  },
  {
      'metadata': {

    "headers": "PHOTOGRAPHS",
    "tags": [
      "date",
      "location",
      "author",
      "source",
      "chain of custody",
      "events depicted"
    ]},
    'text': 'Based on Article 69(4) of the Rome Statute and Rules 63 and 64 of the ICC Rules of Procedure and Evidence, regarding the Court’s authority to rule on the relevance, probative value and admissibility of any evidence, photographs should be accompanied by reliable information on their date, location and events depicted. If the Court does not receive such information, photographs’ relevance to issues in the case and probative value cannot be determined.97 The ICC Trial Chamber in Ntaganda noted that since six photographs brought by the Prosecution were not dated, their relevance and probative value surrounding issues in the case could not be determined.98 It added that when photographs are dated, the parties seeking admission should provide evidence from which the Court can conclude that the dates are correct and fall within the temporal scope of the charges.99 The ICC Trial Chamber in Ntaganda also noted that certain dated photographs could have some relevance, including photos dated ambiguously (such as ‘08/07 2003’, which could be interpreted as either the 8th of July or August 7th) or by a range (‘January-February 2003’), but in the absence of any further reliable information as to the date, location and events depicted in the photographs, it could not admit them into evidence due to lack of probative value.'
  },
  {
      'metadata': {

    "headers": "PHOTOGRAPHS",
    "tags": [
      "photo corroboration",
      "testimony verification",
      "eyewitness",
      "photo validation"
    ]},
    "text": 'Where photographic evidence is of poor quality or it is unclear who took them and/or how they were developed, consistent testimonies from credible witnesses who were at the site can corroborate the content of the photographs.103 The ICC Trial Chamber in Ntaganda noted the consistency of evidence from photographs taken from credible witnesses, and the consistent testimony from seven witnesses, with which it was able to satisfy itself that the photographs did indeed depict the aftermath of a massacre.104 Unreliable Expert Testimony. An expert witness’ testimony is unreliable if it is based on conclusions drawn from photographs displaying obvious limitations in terms of reliability.105 In Mladić, the ICTY Trial Chamber was presented with multiple photographs of the allegedly same crater: one was taken initially by a war correspondent during the conflict in the 1990s, and then others were subsequently taken by Defence experts in 2010. The Trial Chamber found the Defence expert’s conclusions drawn from the photographs were unreliable because of the limitations of the photographs in terms of their reliability.106 Firstly, the Chamber found that the photographs did not in fact depict the same crater, nor the same floor tiles which were depicted in the initial photograph.107 Secondly, editing software was used on the Defence expert’s photographs to place each photograph in a vertical position and remove deformations'
  },
  {
  'metadata': {

   "headers": "AERIAL AND SATELLITE IMAGES",
    "tags": [
      "expert report",
      "satellite image",
      "summary submission",
      "voluminous data"
    ]},
    "text": 'Pursuant to Rule 92 bis (A) of the ICTY Rules of Procedure and Evidence, evidence of a witness in the form of a written statement may be admitted in lieu of oral testimony which goes to proof of a matter other than the acts and conduct of an accused as charged in the indictment. An example of the rule’s application would be if evidence in question is of a cumulative nature in that other witnesses will give, or have given, oral testimony of similar facts. This allows investigators to produce summary reports which are derived from multiple sources and aims to give background evidence to the forensic examinations, thereby contextualising and reducing the apparent complexity of their findings.117 ‘To facilitate matters and to speed up the process’,118 the ICTY in Krstić authorised an investigator with the Office of the Prosecutor to testify in a summary form about the findings of forensic experts who had conducted examinations of various grave sites in 1996, 1998 and 1999 ‘associated with the take-over of Srebrenica’.'
  },
  {
      'metadata': {

    "headers": "AERIAL AND SATELLITE IMAGES",
    "tags": [
      "inseparable evidence",
      "satellite corroboration",
      "aerial photo",
      "linked testimony"
    ]},
    "text": 'Pursuant to Rule 92 bis (D) of the ICTY Rules of Procedure and Evidence,120 ‘a Chamber may admit a transcript of evidence given by a witness in proceedings before the Tribunal which goes to proof of a matter other than the acts and conduct of the accused’. Although Rule 92 bis (D) does not explicitly provide for the admission of exhibits admitted during former testimony, these exhibits are admissible pursuant to this rule so long as they form an inseparable and indispensable part of the testimony (whether expert or not).121 Aerial and satellite images are an inseparable and indispensable part of the testimony if the witness discusses them ‘in his or her written statement or transcript and if that written statement would become incomprehensible or have lesser probative value without [the] admission’ of such images.122 Indexes. Aerial and satellite images admitted during former witness testimony should be tendered with an index. The index should indicate the exact title or exhibit number for each former exhibit to identify the exact exhibits from the previous case.123 The ICTY in Blagojević and Jokić postponed the admission of aerial images that had been previously tendered and admitted at the ICTY during related witness testimony of previous ICTY trials until an index ofproposed exhibits could be provided.1'
  }
]


In [ ]:
rag_database = RAGDatabase(collection_name='leiden_guidelines')
formatted_documents = DocumentParser().format_documents_for_chroma(leiden_documents)
# print(formatted_documents)
embeddings_processor = EmbeddingsProcessor('sentence-transformers/all-MiniLM-L6-v2')
chroma_documents = DocumentParser().parse_for_chroma(formatted_documents, embeddings_processor)
rag_database.store_documents(chroma_documents)

query_embeddings = embeddings_processor.create_embeddings("who painted the Mona Lisa?")
search_results = rag_database.query_with_embeddings(query_embeddings)

import json
formatted_json = json.dumps(search_results, indent=4)
print(formatted_json)



In [ ]:
query_embeddings = embeddings_processor.create_embeddings('tell me about formal submission of a video transcripts and translation')
# query_embeddings = embeddings_processor.create_embeddings("Who is scooby doo?")
# query_embeddings = embeddings_processor.create_embeddings("tell me about satellite imagery")


metadata_query = MetadataQuery().createFilters(headers=['VIDEOS'])

print(f'metadata_query: {metadata_query}')
rag_database = RAGDatabase(collection_name='leiden_guidelines')

search_results = rag_database.query_with_embeddings(query_embeddings, metadata_query)

import json
formatted_json = json.dumps(search_results, indent=4)
print(formatted_json)

In [ ]:
query_embeddings = embeddings_processor.create_embeddings('tell me about formal submission of a video transcripts and translation')
# query_embeddings = embeddings_processor.create_embeddings("Who is scooby doo?")
# query_embeddings = embeddings_processor.create_embeddings("tell me about satellite imagery")


# metadata_query = MetadataQuery().createFilters(headers=['VIDEOS'], tags=['working language', 'court language'])
# metadata_query = MetadataQuery().createFilters(tags=['working language', 'court language'])
metadata_query = MetadataQuery().createFilters(tags=['court language'])

# {'$or': [{'headers': {'$in': ['VIDEOS']}}, {'working language': True}, {'court language': True}]
print(f'metadata_query: {metadata_query}')
rag_database = RAGDatabase(collection_name='leiden_guidelines')

search_results = rag_database.query_with_embeddings(query_embeddings, metadata_query)

import json
formatted_json = json.dumps(search_results, indent=4)
print(formatted_json)


In [ ]:
import pandas as pd

results = rag_database.get_collection('leiden_guidelines')

# Convert to DataFrame
df = pd.DataFrame({
    'id': results['ids'],
    'document': results['documents'],
    'metadata': results['metadatas'],
    'embedding': results['embeddings']  # optional: can be large
})

# Display
df.head(50)

In [ ]:
# results = collection.query(
#     query_texts=["What are the data sources?"],
#     n_results=5,
#     where={"type": "footnote"}
# )
# [
#     {
#         "id": "doc1",
#         "text": "Main article content.",
#         "metadata": {"type": "main", "footnote_ids": ["fn1", "fn2"]}
#     },
#     {
#         "id": "fn1",
#         "text": "Footnote 1: background data",
#         "metadata": {"type": "footnote", "parent": "doc1"}
#     },
#     {
#         "id": "fn2",
#         "text": "Footnote 2: references",
#         "metadata": {"type": "footnote", "parent": "doc1"}
#     }
# ]